# Lesson 13 - Agent Memory


## Setup

Dis notebook show how to build travel booking agent wey get **persistent memory** using **Microsoft Agent Framework** (MAF).

You go learn how different kain agent memory — working, short-term, and long-term — dey affect how agent dey keep and use info across conversations.

**Prerequisites:**
- Microsoft Foundry project wey get deployed chat model (e.g. `gpt-5-mini`).
- You don login with Azure CLI — run `az login` for your terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — your Microsoft Foundry project endpoint.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Kain Agent Memory Dem

AI agents fit use different kain memory, each one get hin own kpai kpai wok:

### Working Memory
Di koko konversation thread — na di messages wey dem dey exchange for one session. Di agent fit look back di earlier messages for di same thread to make sure say everything dey make sense. For MAF e dey create like dis via **`agent.create_session()`**, wey e go return `AgentSession`.

### Short-Term Memory
Information wey go last only for di time of one task or session but no go store gidigba. Example be say, di agent fit gather facts during one multi-turn planning konversation, den use am to produce di final itinerary.

### Long-Term Memory
Preferences and facts wey go last **across sessions**. If person don come before, e no suppose repeat im dietary restrictions or travel style. Long-term memory na usually backed by external store — like database, file, or vector index — wey dem go show to di agent via tools.


## Working Memory wit Sessions

Di simplest form of memory na di conversation session. When you pass di same session (wey be create via `agent.create_session()`) to successive `agent.run()` calls, di agent go fit see di full history of dat conversation and e fit remember tins wey dem talk before.

Make we create travel agent and show how working memory dey work.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Di agent rightly remember di budget becos di two messages dem get di same session. Dis na **working memory** — e dey only last during di session time.

### Wetin dey happen if new thread start?

If we create **new** session, di agent no get memory of di previous talk:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Long-Term Memory Pattern

To remember user preferences **across sessions**, we need a persistent store wey dey outside di conversation thread. Di agent go dey use **tools** — functions wey e fit call to save and fetch information.

Below we go implement one simple in-memory preference store (for production you for take database or vector index hold am) and show am as tools wey di agent fit use.

### Architecture
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — First-time user books an anniversary trip

Sarah waka come for di first time. Di agent suppose store dia preferences through di tools dem and use am recommend hotels.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenario 2 — Sarah show bek say weeks don pass

Sarah start **new kain thread** (wey dey simulate new session). Di working memory empty, but di long-term preference store still get her information. Di agent suppose fetch am come use am take personalize di recomendation dem.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Summary

For dis lesson, you learn three kain agent memory and how to use dem wit di Microsoft Agent Framework:

| Memory Type | MAF Mechanism | Lifetime |
|---|---|---|
| **Working** | `agent.create_session()` | One-time talk |
| **Short-term** | Context wey gather inside one thread | One task / session |
| **Long-term** | External place wey you fit access wit `@tool` functions | Across many sessions |

### Key Takeaways
1. **`agent.create_session()`** dey give working memory — di agent fit see all di conversation history for one session.
2. **New sessions go lose context** — without long-term memory, di agent no go fit remember wetin dem talk before.
3. **`@tool` functions** na bridge — dem allow di agent to save and find information from one place wey dey keep am for long.
4. **Personalization go beta as time dey go** — di more preferences wey dem save, di better di agent go dey recommend.

### Real-World Applications
- **Customer Service**: Make customer history and their preferences no forget
- **Personal Assistants**: Keep context from day to day or week to week
- **Healthcare**: Track patient information plus wetin dem like
- **E-commerce**: Personalized shopping based on history

### Next Steps
- Change di in-memory dict put with database or vector store (like Azure AI Search)
- Add memory expiration for information wey suppose dey valid for short time
- Build multi-agent systems wey get shared memory
- Look into di Cognee notebook for knowledge-graph-backed memory


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
